In [7]:
import cv2 as cv
import numpy as np


# Read the video and its Information
cap = cv.VideoCapture('Q_three.AVI')
i = 0
n_frame = int(cap.get(7))
fps = int(cap.get(5))
h = int(cap.get(4))
w = int(cap.get(3))
win = np.zeros(( h , w , n_frame))
rgbwin = np.zeros(( h , w , 3 , n_frame))
while(cap.isOpened()):
  ret, frame = cap.read()
  if ret == True:
    win[: , : , i] = frame[ : , : , 0 ]
    rgbwin[: , : , : , i] = frame[ : , : , : ]
    i+=1
    if cv.waitKey(25) & 0xFF == ord('q'):
      break
  else: 
    break

rgbwin = np.uint8(rgbwin)

# Find video backgound
background = np.zeros((h , w))
for i in np.arange(n_frame):
    background = win[: , : , i] + background

background = background/n_frame

cv.imshow("Background.png" , np.uint8(background))

# Find The difference between video and Its background
diff = np.zeros(( h , w , n_frame))
for i in np.arange(n_frame):
    diff[: , : , i] = win[: , : , i] - background
# Function for mapping to range of (0 , 255)
def map_0_255(arr):
    #minr = arr.min()
    #maxr = arr.max()
    #d = 0 - minr
    #marr = np.floor((arr+d)*(255/(maxr+d)))
    marr=np.floor(255*(arr-arr.max())/(arr.max()-arr.min())+255)
    marr = marr.astype(np.uint8)
    return marr

#The thresholding on diffrences
diff = np.uint8(map_0_255(diff))
gray_diff = np.zeros(( h , w , n_frame))
rgb_diff = np.zeros(( h , w , 3 , n_frame))
for i in np.arange(n_frame):
    rett , gray_diff[: , : , i] = cv.threshold( diff[: , : , i] , 95 , 255 , cv.THRESH_BINARY)
    gray_diff[: , : , i] = 255 - gray_diff[: , : , i]
gray_diff = np.uint8(gray_diff)

# Show Object Movment by red color
result= np.zeros(( h , w , 3 , n_frame))
result[: , : , : , :]  = rgbwin[: , : , : , :]

for i in np.arange(n_frame):
    result[: , : , 0 , i] = result[: , : , 0 , i] *((255-gray_diff[: , : , i])//255)
    result[: , : , 1 , i] = result[: , : , 1 , i] *((255-gray_diff[: , : , i])//255)
    result[: , : , 2 , i] = result[: , : , 2 , i] *((255-gray_diff[: , : , i])//255)
    result[: , : , 2 , i] = result[: , : , 2 , i] +gray_diff[: , : , i]
result = map_0_255(result)
#result=result.astype(np.uint8)
#cv.imshow("sample",((255-gray_diff[: , : , i])))
cv.imshow("Res.png" , result[: , : , : , 50])

# Make video from result of frame
out = cv.VideoWriter('Q_three_Result.avi',cv.VideoWriter_fourcc(*'DIVX'), fps , (w , h))
for i in np.arange(n_frame):
    out.write(result[: , : , : , i])
out.release()

# Play the Result video
cap = cv.VideoCapture('Q_three_Result.avi')

# Check if camera opened successfully
if (cap.isOpened()== False): 
  print("Error opening video stream or file")

# Read until video is completed
while(cap.isOpened()):
  # Capture frame-by-frame
  ret, frame = cap.read()
  if ret == True:

    # Display the resulting frame
    cv.imshow('Frame',frame)

    # Press Q on keyboard to  exit
    if cv.waitKey(25) & 0xFF == ord('q'):
      break

  # Break the loop
  else: 
    break

# When everything done, release the video capture object
cap.release()

# Closes all the frames
cv.waitKey(0)
cv.destroyAllWindows()
